# 2 - WriteStreak Ingestion and Cleaning

Outputs (in `data/processed/`):
- `writestreak_posts_base.csv`
- `writestreak_posts_final_clean.csv`

If comments and comments indexing is needed, this file is the file to adjust

In [9]:
from __future__ import annotations

from pathlib import Path
import sys, json, hashlib, re

import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()

sys.path.insert(0, str(REPO_ROOT / "src"))
from l2affect.utils.config import load_config, resolve
cfg = load_config(REPO_ROOT / ".." / "configs" / "config.yaml")
processed_dir = resolve(cfg["paths"]["processed_dir"])
processed_dir.mkdir(parents=True, exist_ok=True)
posts_dir = resolve(cfg["inputs"]["writestreak_posts_jsonl_dir"])

In [10]:
def stable_user_id(author: str) -> str:
    h = hashlib.sha256(author.encode("utf-8")).hexdigest()
    return h[:16]

def is_removed(text: str) -> bool:
    t = (text or "").strip().lower()
    return t == "" or t in {"[deleted]", "[removed]"}

def read_jsonl_files(folder: Path) -> list[dict]:
    files = sorted(folder.glob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No .jsonl files found in: {folder}")
    rows = []
    for fp in files:
        with fp.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                r["_source_file"] = fp.name
                rows.append(r)
    return rows

def ingest_posts(posts_dir: Path) -> pd.DataFrame:
    raw = read_jsonl_files(posts_dir)
    rows = []
    for r in raw:
        post_id = r.get("id")
        author = str(r.get("author", ""))
        created_utc = r.get("created_utc", None)

        title = r.get("title", "") or ""
        selftext = r.get("selftext", "") or ""
        if is_removed(selftext) and (title.strip() == ""):
            continue

        rows.append({
            "post_id": post_id,
            "post_fullname": f"t3_{post_id}" if post_id else None,
            "author": author,
            "user_id": stable_user_id(author),
            "created_utc": created_utc,
            "created_at": pd.to_datetime(created_utc, unit="s", utc=True) if created_utc else pd.NaT,
            "title": title,
            "selftext": selftext,
            "subreddit": r.get("subreddit"),
            "permalink": r.get("permalink"),
            "score": r.get("score"),
            "num_comments": r.get("num_comments"),
            "source_file": r.get("_source_file"),
        })

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["post_id"]).drop_duplicates(subset=["post_id"])
    df = df.sort_values(["created_at", "post_id"])
    return df

posts_base = ingest_posts(posts_dir)
posts_base_path = processed_dir / "2-writestreak_posts_base.csv"
posts_base.to_csv(posts_base_path, index=False)

print("Wrote:", posts_base_path)
print("Posts:", len(posts_base), "Users:", posts_base["user_id"].nunique())
posts_base.head()

Wrote: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\2-writestreak_posts_base.csv
Posts: 1284 Users: 57


,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,score,num_comments,source_file
0,i3d4p4,t3_i3d4p4,dzcFrench,142f4dcf1743c538,1596517056,2020-08-04 04:57:36+00:00,r/WriteStreakEN Lounge,A place for members of r/WriteStreakEN to chat...,WriteStreakEN,/r/WriteStreakEN/comments/i3d4p4/rwritestreake...,1,0,202008.jsonl
1,i43wit,t3_i43wit,peche-honey,f56e5f66eabfba77,1596628354,2020-08-05 11:52:34+00:00,Maintain Your Streak and Write Everyday!,Learning languages require practice. Here's th...,WriteStreakEN,/r/WriteStreakEN/comments/i43wit/maintain_your...,1,0,202008.jsonl
2,i43xzn,t3_i43xzn,peche-honey,f56e5f66eabfba77,1596628536,2020-08-05 11:55:36+00:00,Maintain Your Streak and Write Everyday!,Learning languages require practice. Here's th...,WriteStreakEN,/r/WriteStreakEN/comments/i43xzn/maintain_your...,1,7,202008.jsonl
3,i46k9i,t3_i46k9i,peche-honey,f56e5f66eabfba77,1596638592,2020-08-05 14:43:12+00:00,What are your goals for the future?,What would you like to do in the future? Do yo...,WriteStreakEN,/r/WriteStreakEN/comments/i46k9i/what_are_your...,1,0,202008.jsonl
4,i4qs95,t3_i4qs95,peche-honey,f56e5f66eabfba77,1596716313,2020-08-06 12:18:33+00:00,Exotic Pets,**Today's topic: If you could choose any wild...,WriteStreakEN,/r/WriteStreakEN/comments/i4qs95/exotic_pets/,1,0,202008.jsonl


### Remove bot/meta template posts

In [12]:
BOT_PATTERNS = [
    r"the character limit for a tweet is\s*280",
    r"hello writestreakians",
    r"remember, if you didn[’']t write yesterday, your streak number is 1",
    r"if you want to make a comment, write below",
    r"english natives, please help us correct posts",
    r"subjects of the day",
]

base = pd.read_csv(posts_base_path)
selftext = base["selftext"].fillna("").astype(str).str.lower()

mask = pd.Series(False, index=base.index)
for p in BOT_PATTERNS:
    mask = mask | selftext.str.contains(p, regex=True)

filtered = base[~mask].copy()

print("Input:", len(base), "Removed:", len(base[mask]), "Kept:", len(filtered))

Input: 1284 Removed: 33 Kept: 1251


### Create fields: text, tokens, longitudinal 

In [15]:
# Build text
filtered["created_at"] = pd.to_datetime(filtered["created_at"], utc=True, errors="coerce")

title = filtered["title"].fillna("").astype(str).str.strip()
body = filtered["selftext"].fillna("").astype(str).str.strip()

filtered["text"] = np.where(title.ne(""), (title + "\n\n" + body).str.strip(), body)
filtered = filtered[filtered["text"].fillna("").astype(str).str.strip().ne("")].copy()

# Tokenize
token_re = re.compile(r"[A-Za-z']+")
filtered["tokens"] = filtered["text"].astype(str).str.lower().map(lambda t: token_re.findall(t))
filtered["n_tokens"] = filtered["tokens"].map(len)
filtered["n_chars"] = filtered["text"].map(len)

# Longitudinal indices per user
filtered = filtered.sort_values(["user_id", "created_at", "post_id"]).copy()
filtered["post_index"] = filtered.groupby("user_id").cumcount() + 1
first_time = filtered.groupby("user_id")["created_at"].transform("min")
filtered["days_since_first"] = (filtered["created_at"] - first_time).dt.days
filtered["user_post_count"] = filtered.groupby("user_id")["post_id"].transform("count")

# Make tokens CSV-friendly
filtered["tokens"] = filtered["tokens"].map(json.dumps)

final_path = processed_dir / "2-writestreak_posts_final_clean.csv"
filtered.to_csv(final_path, index=False)

print("Wrote:", final_path)
print("Posts:", len(filtered), "Users:", filtered["user_id"].nunique())
filtered.head()

Wrote: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\2-writestreak_posts_final_clean.csv
Posts: 1251 Users: 56


,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,score,num_comments,source_file,text,tokens,n_tokens,n_chars,post_index,days_since_first,user_post_count
797,kt7xl7,t3_kt7xl7,Namatl,00052db50867a4da,1610129063,2021-01-08 18:04:23+00:00,Streak 1 Writing Prompt,"""I was walking to the store in the evening, th...",WriteStreakEN,/r/WriteStreakEN/comments/kt7xl7/streak_1_writ...,1,2,202101.jsonl,"Streak 1 Writing Prompt\n\n""I was walking to t...","[""streak"", ""writing"", ""prompt"", ""i"", ""was"", ""w...",142,721,1,0,2
808,ku0tf4,t3_ku0tf4,Namatl,00052db50867a4da,1610230734,2021-01-09 22:18:54+00:00,Streak 2 Favorite Singer/Band,Mi favorite singer is Michael. I grew up liste...,WriteStreakEN,/r/WriteStreakEN/comments/ku0tf4/streak_2_favo...,1,1,202101.jsonl,Streak 2 Favorite Singer/Band\n\nMi favorite s...,"[""streak"", ""favorite"", ""singer"", ""band"", ""mi"",...",126,701,2,1,2
144,j3vkx8,t3_j3vkx8,Stylelike,000a672e1864051f,1601649246,2020-10-02 14:34:06+00:00,Streak 1: My bike.,"In May, I bought my first motorcycle; my first...",WriteStreakEN,/r/WriteStreakEN/comments/j3vkx8/streak_1_my_b...,1,3,202010.jsonl,"Streak 1: My bike.\n\nIn May, I bought my firs...","[""streak"", ""my"", ""bike"", ""in"", ""may"", ""i"", ""bo...",94,479,1,0,4
154,j4p7n0,t3_j4p7n0,Stylelike,000a672e1864051f,1601768545,2020-10-03 23:42:25+00:00,Streak 2: First aids,I think that learn first aids is indispensable...,WriteStreakEN,/r/WriteStreakEN/comments/j4p7n0/streak_2_firs...,1,4,202010.jsonl,Streak 2: First aids\n\nI think that learn fir...,"[""streak"", ""first"", ""aids"", ""i"", ""think"", ""tha...",65,350,2,1,4
159,j5ct7h,t3_j5ct7h,Stylelike,000a672e1864051f,1601870579,2020-10-05 04:02:59+00:00,Streak 3: What would I do with 10 billion doll...,Well... if I can’t expend it in my family or m...,WriteStreakEN,/r/WriteStreakEN/comments/j5ct7h/streak_3_what...,1,3,202010.jsonl,Streak 3: What would I do with 10 billion doll...,"[""streak"", ""what"", ""would"", ""i"", ""do"", ""with"",...",43,244,3,2,4
